# Dependencies and Imports

In [1]:
import gymnasium as gym
import highway_env
from stable_baselines3 import PPO
from tqdm import trange

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from scipy.stats import gaussian_kde

import torch
import imageio

gym.register_envs(highway_env)

Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


# Load Model

In [2]:
MODEL_PATH = "../merge/merge_ppo/PPO_1/final_model"

model = PPO.load(MODEL_PATH)
env = gym.make("merge-v0", render_mode="rgb_array")
print("Model loaded successfully.")
# print("Sanity check: Model was trained on ", model.num_timesteps, " timesteps.")

Model loaded successfully.


/home/jahall21/miniconda3/lib/python3.12/site-packages/stable_baselines3/common/on_policy_algorithm.py:150: UserWarning: You are trying to run PPO on the GPU, but it is primarily intended to run on the CPU when not using a CNN policy (you are using ActorCriticPolicy which should be a MlpPolicy). See https://github.com/DLR-RM/stable-baselines3/issues/1245 for more info. You can pass `device='cpu'` or `export CUDA_VISIBLE_DEVICES=` to force using the CPU.Note: The model will train, but the GPU utilization will be poor and the training might take longer than on CPU.
  warnings.warn(


# Verify Model Has Low Probability of Failure (Direct Sampling)

In [5]:
NUM_EPISODES = 100
successes = 0
failures = 0

collision_positions = []
collision_speeds = []
collision_types = []

for episode in trange(NUM_EPISODES):
    obs, info = env.reset()
    done = truncated = False
    merger = env.unwrapped.road.vehicles[-1]

    while not (done or truncated):
        action, _ = model.predict(obs, deterministic=True)
        obs, reward, done, truncated, info = env.step(action)

    vehicle = env.unwrapped.vehicle
    if getattr(vehicle, 'crashed', False):
        failures += 1
        collision_positions.append(vehicle.position.copy())
        collision_speeds.append(vehicle.speed)
        hit_merger = merger.crashed
        collision_types.append('merger' if hit_merger else 'highway')
    else:
        successes += 1

merger_fails  = collision_types.count('merger')
highway_fails = collision_types.count('highway')
p_fail = failures / NUM_EPISODES

print(f"Successes:      {successes}/{NUM_EPISODES}")
print(f"Failures:       {failures}/{NUM_EPISODES}")
print(f"  Merger:       {merger_fails} ({merger_fails/NUM_EPISODES*100:.2f}%)")
print(f"  Highway:      {highway_fails} ({highway_fails/NUM_EPISODES*100:.2f}%)")
print(f"P(failure) ≈    {p_fail:.4f}  ({p_fail*100:.2f}%)")

100%|██████████| 100/100 [00:19<00:00,  5.16it/s]

Successes:      99/100
Failures:       1/100
  Merger:       0 (0.00%)
  Highway:      1 (1.00%)
P(failure) ≈    0.0100  (1.00%)


# Fuzzing Helper Functions

In [ ]:
NOMINAL_DELAY_PROB = 0.01  # nominal distribution: small baseline delay probability

def run_rollout(env, model, delay_prob=0.01, nominal_delay_prob=NOMINAL_DELAY_PROB, record_video=False):
    obs, info = env.reset()
    done = truncated = False

    logp_nominal  = 0.0
    logp_proposal = 0.0

    frames = []
    prev_action = None

    merger = env.unwrapped.road.vehicles[-1]

    while not (done or truncated):
        if record_video:
            frames.append(env.render())

        intended_action, _ = model.predict(obs, deterministic=True)

        delayed = False
        if prev_action is not None:
            # delay only possible after first step
            if np.random.rand() < delay_prob:
                action = prev_action
                delayed = True
            else:
                action = intended_action

            # accumulate log-probs only after first step
            if prev_action is not None:
                if delayed:
                    logp_proposal += np.log(delay_prob)
                    logp_nominal  += np.log(nominal_delay_prob)
                else:
                    logp_proposal += np.log(1 - delay_prob)
                    logp_nominal  += np.log(1 - nominal_delay_prob)
        else:
            action = intended_action  # first step: no delay possible, no log-prob contribution

        prev_action = action
        obs, reward, done, truncated, info = env.step(action)

    vehicle = env.unwrapped.vehicle
    failed = bool(getattr(vehicle, "crashed", False))
    failure_type = None
    if failed:
        failure_type = "merger" if merger.crashed else "highway"

    return failed, failure_type, logp_nominal, logp_proposal, frames


def evaluate_proposal(delay_prob, num_episodes, model, env):
    mlf_logp = -np.inf

    failure_logp_nominals  = []
    failure_logp_proposals = []
    is_weights = []

    for _ in trange(num_episodes, desc=f"delay_prob={delay_prob:.3f}", leave=False):
        failed, _, logp_nom, logp_prop, _ = run_rollout(env, model, delay_prob, record_video=False)

        # IS weight: 1[fail] * p_nominal(tau) / p_proposal(tau)
        w = np.exp(logp_nom - logp_prop) if failed else 0.0
        is_weights.append(w)

        if failed:
            failure_logp_nominals.append(logp_nom)
            failure_logp_proposals.append(logp_prop)
            if logp_nom > mlf_logp:
                mlf_logp = logp_nom

    # Re-run once with video only if any failure was found
    mlf_frames = None
    if mlf_logp > -np.inf:
        for _ in range(50):  # try up to 50 times to capture a failure on video
            failed, _, _, _, frames = run_rollout(env, model, delay_prob, record_video=True)
            if failed:
                mlf_frames = frames
                break

    return mlf_logp, mlf_frames, failure_logp_nominals, failure_logp_proposals, is_weights

# Fuzzing per Probability

In [ ]:
delay_probabilities   = np.array([0.01, 0.02, 0.03, 0.05, 0.075, 0.1, 0.2, 0.5])
num_rollouts_per_prob = 100

results = {}
all_failure_logp_noms = []
total_rollouts = 0
total_failures = 0
best_logp = -np.inf
best_frames = None
best_delay_prob = None

for delay_prob in delay_probabilities:
    print(f"\nTesting delay_prob = {delay_prob:.3f}")

    mlf_logp, mlf_frames, fail_noms, fail_props, is_weights = evaluate_proposal(
        delay_prob, num_rollouts_per_prob, model, env
    )

    p_fail_is = np.mean(is_weights)

    results[delay_prob] = {
        "mlf_logp":                mlf_logp,
        "mlf_frames":              mlf_frames,
        "p_fail_is":               p_fail_is,
        "num_failures":            len(fail_noms),
        "num_rollouts":            num_rollouts_per_prob,
        "is_weights":              list(is_weights),
        "failure_logp_nominals":   list(fail_noms),
        "failure_logp_proposals":  list(fail_props),
    }

    all_failure_logp_noms.extend(fail_noms)
    total_rollouts += num_rollouts_per_prob
    total_failures += len(fail_noms)

    if mlf_logp > best_logp:
        best_logp       = mlf_logp
        best_frames     = mlf_frames
        best_delay_prob = delay_prob

    print(f"  Failures:    {len(fail_noms)}/{num_rollouts_per_prob}")
    print(f"  p_fail (IS): {p_fail_is:.5f}")
    print(f"  MLF logp:    {mlf_logp:.3f}" if mlf_logp != -np.inf else "  MLF logp:    -inf")

print(f"\nTotal rollouts: {total_rollouts}")
print(f"Total failures: {total_failures}")
print(f"Best delay_prob: {best_delay_prob}  (MLF logp = {best_logp:.3f})" if best_logp != -np.inf else f"Best delay_prob: {best_delay_prob}  (MLF logp = -inf)")

# Plot: IS p_fail Estimate vs Delay Probability

### Plot 1 — Importance Sampling p_fail estimate vs delay probability

In [ ]:
GROUND_TRUTH_P_FAIL = 0.01994  # 100k direct sampling result

probs          = sorted(results.keys())
p_fail_is_vals = [results[p]["p_fail_is"] for p in probs]
raw_fail_rates = [results[p]["num_failures"] / results[p]["num_rollouts"] for p in probs]

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(probs, p_fail_is_vals, marker='o', label="IS estimate (fuzzing)")
ax.plot(probs, raw_fail_rates,  marker='s', linestyle='--', label="Raw failure rate (biased)")
ax.axhline(GROUND_TRUTH_P_FAIL, color='red', linestyle=':', label=f"Ground truth baseline ({GROUND_TRUTH_P_FAIL:.4f})")
ax.set_title("IS $p_{fail}$ Estimate vs Delay Probability")
ax.set_xlabel("Delay Probability (proposal)")
ax.set_ylabel("$p_{fail}$ estimate")
ax.legend()
ax.grid(True)
plt.tight_layout()
plt.show()

### Plot 2 — MLF log-likelihood vs delay probability (John had this before)

In [ ]:
mlf_logps = [results[p]["mlf_logp"] for p in probs]
mlf_plot  = [v if v != -np.inf else float('nan') for v in mlf_logps]

fig, ax = plt.subplots(figsize=(7, 4))
ax.scatter(probs, mlf_plot, s=80, zorder=5)
ax.set_title("Most Likely Failure Log-Likelihood vs Delay Probability")
ax.set_xlabel("Delay Probability")
ax.set_ylabel("MLF log-likelihood (under nominal)")
ax.grid(True)
plt.tight_layout()
plt.show()

### Plot 3 — Failure log-likelihood histogram (all proposals pooled)

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
if all_failure_logp_noms:
    ax.hist(all_failure_logp_noms, bins=30)
ax.set_title("Failure Log-Likelihood Distribution (under nominal)")
ax.set_xlabel("Log-likelihood under nominal")
ax.set_ylabel("Count")
ax.text(0.05, 0.95, f"Total rollouts: {total_rollouts}\nTotal failures: {total_failures}",
        transform=ax.transAxes, verticalalignment='top')
plt.tight_layout()
plt.show()

# LOAD RESULTS FROM OVERNIGHT SCRIPT
Run the cells below to load `fuzzing_results.json` produced by `delayed_action_local.py` and regenerate all plots.

In [ ]:
import json, os
import numpy as np
import matplotlib.pyplot as plt

RESULTS_PATH = "fuzzing_results_20k.json"  # update filename as needed

with open(RESULTS_PATH) as f:
    raw = json.load(f)

# Parse — keys are strings in JSON, convert back to float
loaded = {float(k): v for k, v in raw.items()}
probs_loaded = sorted(loaded.keys())

print(f"Loaded {len(probs_loaded)} delay probs: {probs_loaded}")
for dp in probs_loaded:
    r = loaded[dp]
    print(f"  delay_prob={dp:.3f}  failures={r['num_failures']}/{r['num_rollouts']}  "
          f"p_fail_is={r['p_fail_is']:.5f}  mlf_logp={r['mlf_logp']}")

In [ ]:
## IS vs Direct Sampling Convergence
# ── CONFIG ───────────────────────────────────────────────────────────────────
DELAY_PROB        = 0.02   # which IS proposal to plot
SKIP              = 500    # skip first N rollouts (zoomed view)

NOMINAL_FILE       = "delayed_action_100k_nominal.json"
NOMINAL_DELAY_PROB = 0.01  # nominal delay probability used in that run
# ─────────────────────────────────────────────────────────────────────────────

def running_stats(weights):
    w   = np.array(weights, dtype=float)
    n   = np.arange(1, len(w) + 1, dtype=float)
    cs  = np.cumsum(w);  cs2 = np.cumsum(w ** 2)
    mean = cs / n
    var  = np.maximum((cs2 - cs**2 / n) / np.maximum(n - 1, 1), 0)
    se   = np.sqrt(var / n)
    return mean, mean - 1.96 * se, mean + 1.96 * se

def style_ax_dark(ax, fig):
    bg = '#1a1a2e'
    fig.patch.set_facecolor(bg);  ax.set_facecolor(bg)
    for spine in ax.spines.values(): spine.set_edgecolor('#555')
    ax.tick_params(colors='white')
    ax.xaxis.label.set_color('white');  ax.yaxis.label.set_color('white')
    ax.title.set_color('white');  ax.grid(True, color='#333', linewidth=0.7)

# IS curve
is_mean, is_lo, is_hi = running_stats(loaded[DELAY_PROB]["is_weights"])
is_ep = np.arange(1, len(is_mean) + 1)

# Direct sampling curve from nominal 100k run (proposal == nominal, so weights are 0/1)
with open(NOMINAL_FILE) as f:
    nominal_data = json.load(f)
direct_weights = nominal_data[str(NOMINAL_DELAY_PROB)]["is_weights"]
direct_mean, direct_lo, direct_hi = running_stats(direct_weights)
direct_ep = np.arange(1, len(direct_mean) + 1)
direct_pf = direct_mean

# y-axis range from both curves after SKIP
y_vals = np.concatenate([is_mean[SKIP:], direct_pf[SKIP:]])
y_pad  = max((y_vals.max() - y_vals.min()) * 0.3, 0.005)
y_lo   = max(0, y_vals.min() - y_pad)
y_hi   = y_vals.max() + y_pad

fig, ax = plt.subplots(figsize=(12, 4))
style_ax_dark(ax, fig)

# IS
ax.fill_between(is_ep[SKIP:], is_lo[SKIP:], is_hi[SKIP:], alpha=0.25, color='dodgerblue')
ax.plot(is_ep[SKIP:], is_mean[SKIP:], color='dodgerblue', lw=1.2,
        label=f'IS delay_prob={DELAY_PROB}  (final={is_mean[-1]:.5f})')

# Direct sampling
ax.fill_between(direct_ep[SKIP:], direct_lo[SKIP:], direct_hi[SKIP:], alpha=0.25, color='red')
ax.plot(direct_ep[SKIP:], direct_pf[SKIP:], color='red', lw=1.2,
        label=f'Direct sampling delay_prob={NOMINAL_DELAY_PROB}  (final={direct_pf[-1]:.5f})')

ax.set_ylim(y_lo, y_hi)
ax.set_xlim(SKIP, max(is_ep[-1], direct_ep[-1]))
ax.set_xlabel('Rollout')
ax.set_ylabel('{fail}$ estimate')
ax.set_title(f'IS (delay_prob={DELAY_PROB}) vs Direct Sampling (delay_prob={NOMINAL_DELAY_PROB}) Convergence', fontsize=12)
ax.legend(facecolor='#222', labelcolor='white', fontsize=9)
plt.tight_layout()
plt.savefig("plot_is_vs_direct.png", dpi=150,
            bbox_inches='tight', facecolor=fig.get_facecolor())
plt.show()

In [ ]:
## IS p_fail Convergence Over Rollouts (per delay_prob)
GROUND_TRUTH_P_FAIL = 0.01994

fig, ax = plt.subplots(figsize=(10, 5))

for dp in probs_loaded:
    weights = np.array(loaded[str(dp)]["is_weights"])
    N = len(weights)
    running_pfail = np.cumsum(weights) / np.arange(1, N + 1)
    ax.plot(running_pfail, label=f"delay_prob={dp}", alpha=0.8)

ax.axhline(GROUND_TRUTH_P_FAIL, color='red', linestyle='--', linewidth=2,
           label=f"Direct MC ground truth ({GROUND_TRUTH_P_FAIL:.4f})")
ax.set_xlabel("Number of Rollouts")
ax.set_ylabel("IS $p_{fail}$ estimate")
ax.set_title("IS $p_{fail}$ Convergence vs Direct MC Ground Truth")
ax.legend(fontsize=8, loc="upper right")
ax.grid(True)
plt.tight_layout()
plt.savefig("plot_is_convergence.png", dpi=150)
plt.show()

In [ ]:
## JSON Plot 1 — IS p_fail estimate vs delay probability
GROUND_TRUTH_P_FAIL = 0.01994  # 100k direct sampling result

p_fail_is_vals = [loaded[dp]["p_fail_is"] for dp in probs_loaded]
raw_fail_rates = [loaded[dp]["num_failures"] / loaded[dp]["num_rollouts"] for dp in probs_loaded]

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(probs_loaded, p_fail_is_vals, marker='o', label="IS estimate (fuzzing)")
ax.plot(probs_loaded, raw_fail_rates,  marker='s', linestyle='--', label="Raw failure rate (biased)")
ax.axhline(GROUND_TRUTH_P_FAIL, color='red', linestyle=':', label=f"Ground truth baseline ({GROUND_TRUTH_P_FAIL:.4f})")
ax.set_title("IS $p_{fail}$ Estimate vs Delay Probability")
ax.set_xlabel("Delay Probability (proposal)")
ax.set_ylabel("$p_{fail}$ estimate")
ax.legend()
ax.grid(True)
plt.tight_layout()
plt.savefig("plot_pfail_vs_delay.png", dpi=150)
plt.show()

# Plot: MLF Log-Likelihood vs Delay Probability

In [ ]:
## JSON Plot 2 — MLF log-likelihood vs delay probability
mlf_logps = [loaded[dp]["mlf_logp"] for dp in probs_loaded]
mlf_plot  = [v if v is not None else float('nan') for v in mlf_logps]

fig, ax = plt.subplots(figsize=(7, 4))
ax.scatter(probs_loaded, mlf_plot, s=80, zorder=5)
ax.set_title("Most Likely Failure Log-Likelihood vs Delay Probability")
ax.set_xlabel("Delay Probability")
ax.set_ylabel("MLF log-likelihood (under nominal)")
ax.grid(True)
plt.tight_layout()
plt.savefig("plot_mlf_logp.png", dpi=150)
plt.show()

# Plot: Failure Log-Likelihood Distribution

In [ ]:
## JSON Plot 3 — IS weight distribution (box plot)
weight_data = [loaded[dp]["is_weights"] for dp in probs_loaded]
labels      = [str(dp) for dp in probs_loaded]

fig, ax = plt.subplots(figsize=(9, 4))
ax.boxplot(weight_data, labels=labels, showfliers=False)
ax.set_title("IS Weight Distribution per Delay Probability")
ax.set_xlabel("Delay Probability")
ax.set_ylabel("IS weight  $w = p_{nom}(\\tau)/p_{prop}(\\tau)$")
ax.grid(True, axis='y')
plt.tight_layout()
plt.savefig("plot_is_weights.png", dpi=150)
plt.show()

## JSON Plot 4 — Failure log-likelihood histogram
all_fail_logps = []
for dp in probs_loaded:
    all_fail_logps.extend(loaded[dp]["failure_logp_nominals"])

total_r = sum(loaded[dp]["num_rollouts"] for dp in probs_loaded)
total_f = sum(loaded[dp]["num_failures"] for dp in probs_loaded)

fig, ax = plt.subplots(figsize=(7, 4))
if all_fail_logps:
    ax.hist(all_fail_logps, bins=40)
ax.set_title("Failure Log-Likelihood Distribution (under nominal, all proposals pooled)")
ax.set_xlabel("Log-likelihood under nominal")
ax.set_ylabel("Count")
ax.text(0.05, 0.95, f"Total rollouts: {total_r}\nTotal failures: {total_f}",
        transform=ax.transAxes, verticalalignment='top')
plt.tight_layout()
plt.savefig("plot_failure_logp_hist.png", dpi=150)
plt.show()

# Save Most Likely Failure Video

In [ ]:
if best_frames is not None:
    imageio.mimsave("most_likely_failure.mp4", best_frames, fps=30)
    print(f"Saved most_likely_failure.mp4")
    print(f"Delay Probability: {best_delay_prob}")
    print(f"Log Likelihood:    {best_logp:.4f}")
else:
    print("No failures occurred during fuzzing.")